# Multi-Tenant MCP Orchestrator: Remote Python Integration

This notebook demonstrates how to programmatically integrate the **Multi-Tenant MCP Orchestrator** and its dynamic **HTTP/SSE transport** into a Python application or custom AI agent workflow.

### Setup & Architecture Blueprint:
1. **Dynamic Workspace Provisioning**: We make an authenticated HTTP POST request to `/api/sessions` to spawn a secure, isolated sandbox container on the fly.
2. **Event Loop Subscription**: We subscribe to the Server-Sent Events (SSE) channel to listen for async JSON-RPC responses from the agent container.
3. **Dual Tunnel Handshake**: We perform the standardized MCP initialization handshake over the dynamic client message posting endpoint.
4. **JSON-RPC Remote Tool Calls**: We invoke filesystem and terminal tools (`write_file`, `run_bash`) remotely and inspect their outputs.

## 1. Environment Configuration
First, we load variables from our `.env` configuration file to locate the Orchestrator gateway URL and bearer API key.

In [ ]:
import os
import asyncio
import json
import queue
import threading
from pathlib import Path
import httpx
from dotenv import load_dotenv

# Load environment variables from parent workspace directory
load_dotenv(Path("..") / ".env")

GATEWAY_URL = "http://localhost:8000"
API_KEY = "super-secret-gateway-key"

print(f"Orchestrator URL target: {GATEWAY_URL}")
print(f"Authorization Token configured: {bool(API_KEY)}")

## 2. Request an Isolated Workspace Session
We send an authenticated request to the Orchestrator to request a dynamic session. Under the hood, this instructs the spawner to pull the latest agent image and boot up a new unprivileged container within a virtual bridge network.

In [ ]:
headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}

print(f"Creating isolated workspace session...")
with httpx.Client() as client:
    res = client.post(f"{GATEWAY_URL}/api/sessions", headers=headers, timeout=20.0)
    assert res.status_code == 200, f"Failed: {res.text}"
    session_data = res.json()
    session_id = session_data["session_id"]
    print(f"\u2705 Session successfully provisioned: {session_id}")

## 3. Establish SSE Tunnel & Discovered Endpoint
To establish the full-duplex tunnel, we start a background thread that listens to the Server-Sent Events (SSE) stream. 

Upon connecting, the Orchestrator automatically broadcasts an `endpoint` event containing the unique message-posting endpoint (`/mcp/{session_id}/?session_id={internal_token}`) reserved for this session. We intercept this event to configure our posting destination.

In [ ]:
sse_events = queue.Queue()
stop_sse = threading.Event()
post_url_container = [None] # Shared container to hold discovered endpoint

def consume_sse_stream():
    try:
        with httpx.Client(timeout=None) as sse_client:
            with sse_client.stream("GET", f"{GATEWAY_URL}/mcp/{session_id}/sse", headers=headers) as stream:
                current_event = None
                for line in stream.iter_lines():
                    if stop_sse.is_set():
                        break
                    if not line:
                        continue
                    if line.startswith("event:"):
                        current_event = line.replace("event:", "").strip()
                    elif line.startswith("data:"):
                        data = line.replace("data:", "").strip()
                        sse_events.put({"event": current_event, "data": data})
                        current_event = None
    except Exception as e:
        print(f"[SSE ERROR] {e}")

# Start the listener thread
sse_thread = threading.Thread(target=consume_sse_stream, daemon=True)
sse_thread.start()
print("\ud83d? Background SSE stream consumer started.")

# Discover the posting endpoint
print("Waiting for endpoint assignment event...")
for _ in range(50):
    try:
        ev = sse_events.get(timeout=0.2)
        if ev["event"] == "endpoint":
            post_url_container[0] = f"{GATEWAY_URL}{ev['data']}"
            break
    except queue.Empty:
        continue

POST_URL = post_url_container[0]
assert POST_URL is not None, "Failed to receive endpoint event from SSE stream"
print(f"\u2705 Discovered target POST URL: {POST_URL}")

## 4. Perform Standard MCP Handshake
We send an `initialize` JSON-RPC request to the discovered POST URL, then verify the initialize response streamed back on the SSE channel. Once received, we complete the handshake by sending the `notifications/initialized` event.

In [ ]:
with httpx.Client() as client:
    # 1. Send MCP initialize
    print("Sending MCP initialization request...")
    init_payload = {
        "jsonrpc": "2.0",
        "method": "initialize",
        "params": {
            "protocolVersion": "2024-11-05",
            "capabilities": {},
            "clientInfo": {
                "name": "notebook-client",
                "version": "1.0.0"
            }
        },
        "id": 100
    }
    init_res = client.post(POST_URL, headers=headers, json=init_payload, timeout=20.0)
    assert init_res.status_code in [200, 202]
    
    # 2. Wait for initialize response on SSE stream
    init_response = None
    for _ in range(50):
        try:
            ev = sse_events.get(timeout=0.2)
            if ev["event"] == "message":
                data = json.loads(ev["data"])
                if data.get("id") == 100:
                    init_response = data
                    break
        except queue.Empty:
            continue
            
    assert init_response is not None, "Failed to receive initialize response"
    print(f"\u2705 Initialize Handshake approved. Capabilities: {init_response['result']['capabilities'].keys()}")

    # 3. Send initialized notification
    print("Sending initialized notification...")
    initialized_payload = {
        "jsonrpc": "2.0",
        "method": "notifications/initialized"
    }
    initialized_res = client.post(POST_URL, headers=headers, json=initialized_payload, timeout=20.0)
    assert initialized_res.status_code in [200, 202]
    print("\u2705 Full-Duplex MCP handshake successfully completed!")

## 5. Remotely Execute Tools
We can now remotely invoke tools inside our isolated workspace. Let's start by writing a file using the `write_file` tool.

In [ ]:
with httpx.Client() as client:
    print("Calling write_file tool remotely...")
    rpc_payload = {
        "jsonrpc": "2.0",
        "method": "tools/call",
        "params": {
            "name": "write_file",
            "arguments": {
                "filepath": "orchestrator_demo.py",
                "content": "print('Hello from a dynamically spawned unprivileged container!')"
            }
        },
        "id": 101
    }
    
    res = client.post(POST_URL, headers=headers, json=rpc_payload, timeout=20.0)
    assert res.status_code in [200, 202]
    
    # Check response from SSE
    write_response = None
    for _ in range(50):
        try:
            ev = sse_events.get(timeout=0.2)
            if ev["event"] == "message":
                data = json.loads(ev["data"])
                if data.get("id") == 101:
                    write_response = data
                    break
        except queue.Empty:
            continue
            
    assert write_response is not None and "error" not in write_response
    print(f"\u2705 File successfully written: {write_response['result']['content'][0]['text']}")

## 6. Run Terminal Execution
Now, let's invoke `run_bash` remotely to run our script inside the isolated workspace container, executing under an unprivileged user profile.

In [ ]:
with httpx.Client() as client:
    print("Calling run_bash tool remotely...")
    rpc_payload = {
        "jsonrpc": "2.0",
        "method": "tools/call",
        "params": {
            "name": "run_bash",
            "arguments": {
                "command": "python orchestrator_demo.py"
            }
        },
        "id": 102
    }
    
    res = client.post(POST_URL, headers=headers, json=rpc_payload, timeout=20.0)
    assert res.status_code in [200, 202]
    
    # Check response from SSE
    bash_response = None
    for _ in range(50):
        try:
            ev = sse_events.get(timeout=0.2)
            if ev["event"] == "message":
                data = json.loads(ev["data"])
                if data.get("id") == 102:
                    bash_response = data
                    break
        except queue.Empty:
            continue
            
    assert bash_response is not None and "error" not in bash_response
    content = bash_response['result']['content'][0]['text']
    print(f"\u2705 run_bash output received:\n\n{content.strip()}")

## 7. Dynamic Session Clean up
When work is finished, we request the Orchestrator to terminate the session. The Orchestrator tears down the container and deletes the ephemeral host directories instantly, guaranteeing zero state leaks.

In [ ]:
with httpx.Client() as client:
    print(f"Pruning and terminating dynamic session {session_id}...")
    res = client.delete(f"{GATEWAY_URL}/api/sessions/{session_id}", headers=headers, timeout=20.0)
    assert res.status_code == 200
    print(f"\u2705 Dynamic workspace cleanly reaped: {res.json()}")
    
# Clean up background event thread
stop_sse.set()
sse_thread.join(timeout=1.0)
print("SSE client thread terminated. Run complete.")